[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Narinder-2006/narinder-flyrank-internship-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

## 0. Setup (Colab or local)
On Colab this clones the repo (so the starter CSV is available) and moves into it.
Locally it just moves from `work/notebooks/` to the repo root.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Narinder-2006/narinder-flyrank-internship-1"
REPO_DIR = "narinder-flyrank-internship-1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # move from work/notebooks/ to the repo root

print("Working dir:", os.getcwd())

Working dir: /home/claude/repo


# Week 2 — ML Task Framing

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring
**Goal:** map this lane onto the ML loop *before* touching real feature/model code.

## 1. My lane as an ML task (type)

This is fundamentally a **ranking / scoring problem**, built on top of a **binary
classification** sub-task.

- The end deliverable a content strategist needs is not a single yes/no per page — it's an
  **ordered queue**: which pages to look at *first*, second, third, given limited review hours.
  That's ranking.
- But the way I'll get a rankable score is by training a model to solve a simpler classification
  question underneath it — *is this page likely to be declining / worth flagging?* — and using
  the model's predicted **probability** as the ranking signal, not a hard label. That's exactly
  the pattern the repo's own reference pipeline uses: `03_train_model.py` trains a classifier,
  and `final_refresh_score` blends the model's probability with a normalized baseline score
  rather than outputting a bare class.
- It is **not** clustering — I'm not looking for undiscovered groups of similar pages, I have a
  specific outcome in mind (declining vs not) that I want to predict.
- It is **not** pure regression on a continuous number either — although a "how much traffic
  will this page lose" version could be framed as regression, I'm choosing the classification +
  ranking framing because the action a strategist takes ("review this page or not, this week")
  is fundamentally a yes/no decision per page, and probability-based ranking is the cleanest way
  to prioritize many yes/no decisions with limited time.

## 2. Target or proxy

**Starter/beginner proxy** (what the reference pipeline uses, and what I'll start with to get a
working baseline this week):

```
is_declining_label = (trend_direction == "down")
```

This is a **current-window derived bucket**, not a real future outcome — the lane guide is
explicit that this is a beginner proxy, not the ideal capstone target, because a model trained
to predict "is this page currently labeled down" is really just learning to recover a label
that already exists in the current 30-vs-prev-30 window, which is a much easier (and less
useful) problem than actually forecasting the future.

**Stronger target I want to move toward for the real capstone**, once I have the fuller
warehouse data with more history:

```
features from days [-90, -30] (relative to a cutoff)  ->  decline or recovery over the next 30 days
```

i.e. predict a **future** outcome from **past-only** features, with a real time gap between
them. This is the leakage-safe version — Week 2's job for me is naming this target correctly in
words; actually building the future-looking label is Week 3+ work once I have the daily-level
warehouse table (the starter CSV is pre-aggregated to 90-day windows per page, so it doesn't
have enough resolution to build a clean future-vs-past split — the daily fact table does).

## 3. Success metric

Since the real deliverable is a **ranked queue** with a limited review budget (a strategist
looks at maybe the top 20–50 pages a week, not all 30,000+), the metric has to match that
review budget, not just be "accuracy":

- **Primary: Precision@K** (K = 20 and K = 50) — of the top K pages the model/score ranks
  highest, how many are actually declining (by whatever label I've defined)? This directly
  answers "if a strategist trusts my top 20 today, how often are they not wasting their time."
- **Secondary: Average Precision (AP)** — rewards putting the *true* declining pages as close
  to the top as possible across the whole ranking, not just within one fixed K, useful for
  comparing models/baselines overall rather than at one cutoff.
- **Reported alongside both: the base rate** (the plain % of pages that are actually
  `trend_direction == "down"` in the data — I computed this in Week 1: ~54%). A high Precision@K
  only means something if it clearly beats that base rate; if 54% of all pages are declining,
  a Precision@50 of 55% would be barely better than picking pages at random.
- I will **not** lead with plain accuracy — with a fairly balanced-ish label (54/46) accuracy is
  less misleading here than in a rare-event problem, but it still doesn't reflect the
  review-budget-constrained way this output actually gets used, so it's not my headline number.

## 4. The unit of analysis, as a real dataframe

One row = **one page** (`content_id`, scoped within a `client_id`). Loading the real starter
data and showing that shape directly.

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("shape:", df.shape)
print("one row per content_id? ", df["content_id"].is_unique)

unit_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "content_age_days", "days_since_last_update", "trend_direction", "trend_pct",
]
df[unit_cols].head(5)

shape: (30000, 44)
one row per content_id?  True


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,10.6,0.76,187,20,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,20.3,0.05,445,25,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,36.5,0.09,141,20,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,6.2,0.49,463,22,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,44.0,0.13,263,14,down,-34.7


In [3]:
# Sketch the proxy target column described in Section 2
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("target value counts:")
print(df["is_declining_label"].value_counts())
print()
print("base rate (share declining):", round(df["is_declining_label"].mean(), 3))

df[["content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"]].head(8)

target value counts:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

base rate (share declining): 0.542


,content_id,client_id,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,down,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,down,-38.9,1
6,content_9a34b442b552,client_8722616204,down,-92.3,1
7,content_a63219c6e95a,client_19581e27de,stable,0.6,0


**Reading the output above:** the target column is a simple binary flag built off a column
that already exists in the data (`trend_direction`) — that's exactly why Section 2 calls it a
"beginner proxy" and not the real target: a model predicting this label is close to just
re-deriving `trend_direction`, not forecasting anything new. `trend_pct` shown alongside it is
the continuous version of the same signal — useful to sanity-check the label, but it (and
`trend_direction` itself) must **not** be used as a model *feature*, only as the label — using
it as a feature would leak the answer directly into the inputs.

## 5. Why ML beats a fixed rule here

It doesn't, *by itself*, and that's worth saying honestly — the repo's own baseline
(`02_baseline_score.py`) already gets you a decent transparent starting rule with no ML at all:

```
baseline_refresh_score =
  0.40 * visibility_score
+ 0.30 * freshness_risk_score
+ 0.25 * position_opportunity_score
+ 0.05 * depth_gap_score
```

So the honest question isn't "rule vs. ML" in the abstract, it's: **does a learned model beat
that specific, auditable rule, on the same pages, by enough to justify losing some
transparency?** Reasons I expect it can, for this lane specifically:

- The baseline is a fixed linear combination of hand-picked weights (0.40 / 0.30 / 0.25 / 0.05).
  Those weights were chosen once, by a person, and don't adapt if (say) `freshness_risk` turns
  out to matter twice as much for `blog` content as for `product` pages. A model can learn
  **interactions** between features (content type × freshness × position) that a single linear
  weighted sum can't represent.
- With 30,000+ rows and dozens of numeric/categorical features (search volume, CTR, position,
  engagement rate, scroll rate, AI traffic %, tiers...), there's more signal available than four
  hand-picked sub-scores can compactly summarize — a model can use all of it at once.
- I can **measure** whether it's actually worth it, directly: Precision@K and AP, baseline vs.
  model, same split, same data (Section 3). If the model doesn't clearly beat the baseline, the
  honest conclusion is to ship the transparent rule instead — that's a legitimate capstone
  result, not a failure.
- Whatever wins, the **output stays reason-coded** either way (this is a repo-wide rule for this
  lane) — so even the ML version doesn't become a black box; it still has to explain itself in
  the same `stale_visible_page` / `declining_with_demand` / `page_one_decay_risk`-style language
  the baseline already uses.

## 6. Self-check

- [x] Named the ML task type: ranking (via a classification sub-task), not clustering or
      regression, and explained why.
- [x] Named the target/proxy: `is_declining_label = trend_direction == "down"` (starter proxy),
      and the stronger future-looking target I'm moving toward (`prior 90d -> next 30d outcome`).
- [x] Named the success metric: Precision@20/@50 and Average Precision, reported next to the
      base rate (~54% declining, from Week 1) so a high score can't hide behind a high base rate.
- [x] Showed the unit of analysis as a real, loaded dataframe — one row = one `content_id`,
      confirmed unique — and sketched the actual target column with real value counts.
- [x] Explained why ML beats a fixed rule here **conditionally**, not by assertion: the existing
      baseline is a legitimate, transparent competitor, and the model only earns its complexity
      if it beats it on Precision@K on the same held-out split — to be proven in Week 3+, not
      assumed here.
- [x] Tied the output to a real content action: strategist reviews the top-K ranked pages this
      sprint (unchanged from Week 1's framing).
- [ ] Not yet done (Week 3+): actually build the leakage-safe future-looking label from the
      daily warehouse table, train the model, and run the baseline-vs-model comparison.

**Lane status:** still Lane 2, still provisional — nothing here changes that.